# Finetune ds distilled model

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported


model_dir = '/root/autodl-tmp/models/DeepSeek-R1-Distill-Qwen-1.5B'
experiment = 'ds_r1_law_1.5B_exp2'

new_model_local_dir = f'{experiment}_base'
print(f'new model local dir: {new_model_local_dir}')

new_merged_model_local_dir = f'{experiment}_merged'
print(f'new merged model local dir: {new_merged_model_local_dir}')

eval_result_dir = f"{experiment}_eval_result"
print(f'eval result save dir: {eval_result_dir}')


# data
local_sft_data_path = '/root/autodl-tmp/dataset/finetune_processed_train.json'
eval_data_path = '/root/autodl-tmp/dataset/finetune_processed_eval.json'


# finetune hyper parameter
max_seq_length = 2048
dtype = None 
load_in_4bit = True
load_in_8bit, full_finetuning = False, False
learning_rate = 2e-4
num_train_epochs = 1
max_steps = -1

lora_rank = 16
lora_alpha = 16

batch_size = 32

# eval
text2vec_model_path = '/root/autodl-tmp/models/text2vec-base-chinese'
eval_sample_num = 1000 # how many batches to run when eval
eval_max_len = 512 # max len in generate outputs



In [ ]:
prompt_style = """下面是一个法律咨询问题，请提供一个回复来解决咨询问题，不需要提供思考过程。
### 指令：
你是一个法律咨询专家，请回答以下问题，不需要提供思考过程。

### 问题：
{}

### 回复:
{}"""

train_prompt_style = """下面是一个法律咨询问题，请提供一个回复来解决咨询问题，不需要提供思考过程。
### 指令：
你是一个法律咨询专家，请回答以下问题，不需要提供思考过程。

### 问题：
{}

### 回复:
{}"""



# Load model

In [ ]:
%%time

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_dir,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, 
    load_in_8bit = load_in_8bit,
    full_finetuning = full_finetuning,
)
print(f'finish loading model')

peft_model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=lora_alpha,
    lora_dropout=0,  
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print(f'finish loading peft model')

In [ ]:
%%time

question = """
农村宅基地可以继承吗，需要办理什么手续才可以建房
"""

FastLanguageModel.for_inference(model) 
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=eval_max_len,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### 回复:")[1])

In [ ]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    inputs = examples["question"]
    outputs = examples["answer"]
    texts = []
    for inputs, outputs in zip(inputs, outputs):
        text = train_prompt_style.format(inputs, outputs) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

In [ ]:
%%time

dataset = load_dataset("json", data_files=local_sft_data_path, split="train")

processed_dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

print('dataset example')
print(processed_dataset['text'][0])


# Finetune training

In [ ]:
%%time

trainer = SFTTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    train_dataset=processed_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        # Use num_train_epochs = 1, warmup_ratio for full training runs!
        warmup_steps=5,
        max_steps=max_steps,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)



In [ ]:
%%time
trainer_stats = trainer.train()
print('train stats')
print(trainer_stats)
print(f'finish training')


finish training
CPU times: user 59min 46s, sys: 34min 55s, total: 1h 34min 41s
Wall time: 1h 32min 48s


In [ ]:
%%time

question = """
农村宅基地可以继承吗，需要办理什么手续才可以建房
"""

FastLanguageModel.for_inference(peft_model) 
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = peft_model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=eval_max_len,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### 回复:")[1])

In [ ]:
%%time

peft_model.save_pretrained(new_model_local_dir) 
tokenizer.save_pretrained(new_model_local_dir)

peft_model.save_pretrained_merged(new_merged_model_local_dir, tokenizer, save_method = "merged_16bit",)

print(f'finetuned model saved to {new_model_local_dir}, merged model saved to {new_merged_model_local_dir}')

finetuned model saved to ds_r1_law_1.5B_base, merged model saved to ds_r1_law_1.5B_merged
CPU times: user 9.96 s, sys: 5.24 s, total: 15.2 s
Wall time: 15.1 s


# Eval model

In [ ]:
from sentence_transformers import SentenceTransformer


In [ ]:
def _cos_sim(a, b):
    from numpy import dot
    from numpy.linalg import norm
    divider = norm(a) * norm(b)
    if abs(divider) < 1e-6:
        return 0
    return dot(a, b) / divider


def answer_sim(a, b, text2vec_model):
    if len(a) == 0 and len(b) == 0:
        return -1
    if len(a) == 0 or len(b) == 0:
        return -1

    embeddings = text2vec_model.encode([a, b])
    cos_sim = _cos_sim(embeddings[0], embeddings[1])
    return cos_sim

In [ ]:
%%time

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_dir,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, 
    load_in_8bit = load_in_8bit,
    full_finetuning = full_finetuning,
)
print(f'finish loading model')


trained_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=new_merged_model_local_dir,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit,
    full_finetuning=full_finetuning,
)

print(f'loaded trained model from {new_merged_model_local_dir}')


In [ ]:
%%time

text2vec_model = SentenceTransformer(text2vec_model_path)
print(f'loaded text2vec model from {text2vec_model_path}')


In [ ]:
def formatting_eval_prompts_func(example):
    inputs = example["question"]
    outputs = example["answer"]
    text = prompt_style.format(inputs, "")

    example['text'] = text
    return example


def group_batch(batch):
    return {k: [v] for k, v in batch.items()}
    

In [ ]:
%%time

eval_data = load_dataset("json", data_files=eval_data_path, split='train')

eval_data = eval_data.map(
    formatting_eval_prompts_func,
    batched=False,
)

eval_data = eval_data.map(group_batch, batched=True, batch_size=batch_size)

print('eval dataset example')
print(eval_data['text'][0])


In [ ]:
def pred_anwser(eval_data, model, tokenizer, eval_sample_num=5, eval_max_len=512):
    FastLanguageModel.for_inference(model)
    rows = []
    for i, ex in enumerate(eval_data):
        if i >= eval_sample_num:
            break
        print(f'processing batch {i}')
        
        questions = ex['question']
        answers = ex['answer']
    
        inputs = tokenizer(ex['text'], return_tensors="pt", padding=True, truncation=True).to("cuda")
        outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=eval_max_len,
                use_cache=True,
            )
        responses = tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    
        for question, answer, response in zip(questions, answers, responses):
            pred = response.split("### 回复:")[1]
            cos_sim = answer_sim(answer, pred, text2vec_model)
            rows.append({
                'question': question,
                'answer': answer,
                'pred': pred,
                'cos_sim': cos_sim,
            })
    
    eval_result = pd.DataFrame(rows)
    print(f'total {eval_result.shape[0]} records')
    
    return eval_result
    

In [ ]:
%%time

# trained model

trained_df = pred_anwser(eval_data, trained_model, tokenizer, eval_sample_num, eval_max_len)

os.makedirs(eval_result_dir, exist_ok=True)
trained_df.to_parquet(os.path.join(eval_result_dir, 'eval_result_trained.parquet'))

print('finish eval trained model, response average cos sim', trained_df['cos_sim'].mean())



In [ ]:
%%time

# original model

original_df = pred_anwser(eval_data, model, tokenizer, eval_sample_num, eval_max_len)

os.makedirs(eval_result_dir, exist_ok=True)
original_df.to_parquet(os.path.join(eval_result_dir, 'eval_result_original.parquet'))


print('finish eval original model, response average cos sim', original_df['cos_sim'].mean())



processing batch 3


processing batch 4


processing batch 5


processing batch 6


processing batch 7


...

processing batch 496


processing batch 497


processing batch 498


processing batch 499


total 1000 records
finish eval original model, response average cos sim 0.7978497
CPU times: user 22min 42s, sys: 2.84 s, total: 22min 45s
Wall time: 22min 43s


processing batch 3


processing batch 4


processing batch 5


processing batch 6


processing batch 7


...

processing batch 496


processing batch 497


processing batch 498


processing batch 499


total 1000 records
finish eval trained model, response average cos sim 0.76458037
CPU times: user 55min 9s, sys: 5.93 s, total: 55min 15s
Wall time: 55min 10s
